In [1]:
import pandas as pd
import numpy as np

import seaborn as sns

In [2]:
data=pd.read_csv("data[1].csv")
df=pd.DataFrame(data)
df.sample(5)

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
432,2014-05-09 00:00:00,311100.0,4.0,2.25,2130,8078,1.0,0,0,4,1380,750,1977,0,3502 Morris Ave S,Renton,WA 98055,USA
814,2014-05-16 00:00:00,282613.0,2.0,1.00,830,6017,1.0,0,0,4,830,0,1954,1979,16221 Bagley Pl N,Shoreline,WA 98133,USA
401,2014-05-09 00:00:00,565000.0,4.0,2.25,2470,7447,2.0,0,0,3,2470,0,1984,0,4205 NE 169th Ct,Lake Forest Park,WA 98155,USA
1321,2014-05-26 00:00:00,379950.0,3.0,1.50,1690,9144,1.0,0,0,4,1140,550,1956,0,6601 NE 153rd Pl,Kenmore,WA 98028,USA
2729,2014-06-18 00:00:00,330000.0,3.0,2.50,1600,26977,2.0,0,0,3,1600,0,2005,0,32702 NE 202nd St,Duvall,WA 98019,USA


Task 1 – Missing Value Engineering

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   str    
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   str    
 15  city           4600 non-null   str    
 16  statezip       4600 non-null   str    
 17  country        4600 non-null   str    
dtypes: float64(4), int6

Filtering the data

In [4]:
df.isnull().sum()
ddf=df.drop(columns=["yr_renovated","statezip",],inplace=True)

In [5]:
df.isnull().sum()

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
street           0
city             0
country          0
dtype: int64

Insight - > NO missing values 

In [6]:
df.sample(3)

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,street,city,country
3291,2014-06-25 00:00:00,235000.0,4.0,2.00,1570,9415,2.0,0,0,4,1570,0,1984,31845-31899 125th Pl SE,Auburn,USA
3922,2014-07-03 00:00:00,265000.0,3.0,1.00,1020,8610,1.0,0,0,5,1020,0,1959,3704 NE 9th Ct,Renton,USA
3360,2014-06-25 00:00:00,450000.0,4.0,2.75,2310,5650,1.0,0,2,3,1330,980,1952,9853 Arrowsmith Ave S,Seattle,USA


Feature Eng

In [7]:
df["Total_area"]=df["sqft_living"]+df["sqft_lot"]+df["sqft_above"]+df["sqft_basement"]
df.drop(columns=["sqft_living","sqft_lot","sqft_above","sqft_basement"],inplace=True)

In [8]:
df.head(2)

,date,price,bedrooms,bathrooms,floors,waterfront,view,condition,yr_built,street,city,country,Total_area
0,2014-05-02 00:00:00,313000.0,3.0,1.5,1.5,0,0,3,1955,18810 Densmore Ave N,Shoreline,USA,10592
1,2014-05-02 00:00:00,2384000.0,5.0,2.5,2.0,0,4,5,1921,709 W Blaine St,Seattle,USA,16350


House Age

In [9]:
df["date"]=pd.to_datetime(df["date"])
#Extracting year of sold
df["sold_year"]=df["date"].dt.year

In [10]:
df["Age"]=df["sold_year"]-df["yr_built"]
df=df.drop(columns=["date","yr_built","sold_year"])
df.head(3)

,price,bedrooms,bathrooms,floors,waterfront,view,condition,street,city,country,Total_area,Age
0,313000.0,3.0,1.5,1.5,0,0,3,18810 Densmore Ave N,Shoreline,USA,10592,59
1,2384000.0,5.0,2.5,2.0,0,4,5,709 W Blaine St,Seattle,USA,16350,93
2,342000.0,3.0,2.0,1.0,0,0,4,26206-26214 143rd Ave SE,Kent,USA,15807,48


In [11]:
cat_col=["waterfront","view","condition"]
num_col=["floors","bedrooms","Total_area","Age"]

In [12]:
from sklearn.model_selection import train_test_split
x=df.iloc[0:,1:]
y=df["price"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

scaling and encoding(these all are  rating from 5 ismai model 1<2<3<4<5 aisa h toh is liye humne onhot hi lagaya agr reverse hota toh hum ordinal lgatw)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder

col_pipe=Pipeline([
    ("category",OneHotEncoder(sparse_output=False,handle_unknown="ignore"))
])


num_pipe=Pipeline([
    ("sacling",StandardScaler())
])

In [14]:
from sklearn.compose import ColumnTransformer 
preprocessor=ColumnTransformer([
    ("num",num_pipe,num_col),
    ("cat",col_pipe,cat_col)
])

In [20]:
from sklearn.linear_model import LogisticRegression

model_pipe=Pipeline([
    ("pre_process",preprocessor),
    ("model",LogisticRegression())
])

In [22]:
model_pipe.fit(x_train,y_train)


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [ ]:
y_pred=model_pipe.predict(x_test)
from sklearn.metrics import r2_score

In [ ]:
r2_score(y_pred,y_test)

-25.090265556061624

In [ ]:
x_test.shape,x_train.shape

((920, 11), (3680, 11))